<h1 align="center">Deep Learning Models for Promoter Prediction</h1>
<h2 align="center">Evaluating CNNs and Transformers on Genomic Sequences</h2>
<h3 align="center">Data Setup and Preparation </h3>

<div style="margin-bottom: 50px;"></div>
<div style="display: flex; justify-content: center;">
    <table style="border-collapse: collapse; border: 0;">
        <tr>
            <td style="text-align:center; vertical-align:middle; padding:10px 40px 10px 10px; border: 0;">
                <h3 style="margin: 0;">Radek Holik</h3>
                April 2026
            </td>
        </tr>
    </table>
</div>
<br>



### Description
This notebook performs initial data preparation, including extracting genomic sequence data and organizing files for downstream analysis. These steps are executed once and are not required for repeated model training.

### Imports and Paths

In [1]:
from pathlib import Path
import os
import gzip
import shutil

In [2]:
# Paths
project_root = Path("..").resolve()
raw_dir = project_root / "data" / "raw"
genome_dir = raw_dir / "genome"

gz_path = genome_dir / "Homo_sapiens.GRCh38.dna.chromosome.1.fa.gz"
fa_path = genome_dir / "chr1.fa"

# Helper for printing
def rel(path):
    return Path(os.path.relpath(path, project_root))

print("Raw data folder:", rel(raw_dir).as_posix())
print("Genome folder:", rel(genome_dir).as_posix())
print("Genome file (.gz):", rel(gz_path).as_posix())
print("Genome file (.fa):", rel(fa_path).as_posix())

Raw data folder: data/raw
Genome folder: data/raw/genome
Genome file (.gz): data/raw/genome/Homo_sapiens.GRCh38.dna.chromosome.1.fa.gz
Genome file (.fa): data/raw/genome/chr1.fa


In [3]:
# Create folder if needed
genome_dir.mkdir(parents=True, exist_ok=True)
print("Genome folder is ready.")

Genome folder is ready.


### Unzipping

In [4]:
# Unzip genome file once
if not gz_path.exists():
    raise FileNotFoundError(f"Compressed genome file not found: {rel(gz_path)}")

if not fa_path.exists():
    print("Unzipping genome file...")
    with gzip.open(gz_path, "rb") as f_in:
        with open(fa_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"Created: {rel(fa_path)}")
else:
    print(f"Unzipped file already exists: {rel(fa_path).as_posix()}")

Unzipping genome file...
Created: data\raw\genome\chr1.fa


In [5]:
# Load FASTA file
print("Reading FASTA file...")

with open(fa_path, "r") as f:
    header = f.readline().strip()
    sequence = "".join(line.strip() for line in f)

print("FASTA file loaded.")

Reading FASTA file...
FASTA file loaded.


### Data Summary

In [6]:
# Find first real DNA base
print("Searching for the first non-N base...")

first_real_idx = next((i for i, base in enumerate(sequence) if base in {"A", "C", "G", "T"}), None)

if first_real_idx is None:
    raise ValueError("No real DNA bases (A/C/G/T) were found in the sequence.")

print(f"First non-N base found at position: {first_real_idx:,}")

Searching for the first non-N base...
First non-N base found at position: 10,000


In [7]:
# Preview
print("Header:")
print(header)
print()

print("First 100 characters of the raw sequence:")
print(sequence[:100])
print()

print("First 100 real DNA bases after leading Ns:")
print(sequence[first_real_idx:first_real_idx + 100])
print()

# File size
file_size_bytes = fa_path.stat().st_size
file_size_mb = file_size_bytes / (1024 ** 2)

# Final summary
total_length = len(sequence)
n_count = sequence.count("N")
acgt_count = sum(sequence.count(base) for base in "ACGT")

print("========== DATASET SUMMARY ==========")
print(f"Chromosome file: {rel(fa_path).as_posix()}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Total sequence length: {total_length:,} bases")
print(f"Number of N bases: {n_count:,}")
print(f"Number of A/C/G/T bases: {acgt_count:,}")
print(f"First real DNA base position: {first_real_idx:,}")
print("====================================")

Header:
>1 dna:chromosome chromosome:GRCh38:1:1:248956422:1 REF

First 100 characters of the raw sequence:
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN

First 100 real DNA bases after leading Ns:
TAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAAC

========== DATASET SUMMARY ==========
Chromosome file: data/raw/genome/chr1.fa
File size: 241.38 MB
Total sequence length: 248,956,422 bases
Number of N bases: 18,475,410
Number of A/C/G/T bases: 230,481,012
First real DNA base position: 10,000
